<a href="https://www.kaggle.com/code/avikdas567/spaceship-titanic?scriptVersionId=294144520" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# Spaceship Titanic
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from lightgbm import LGBMClassifier

# =====================
# Load Data
# =====================
train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test  = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

# =====================
# Combine for preprocessing
# =====================
train['is_train'] = 1
test['is_train'] = 0
test['Transported'] = np.nan

df = pd.concat([train, test], ignore_index=True)

# =====================
# Feature Engineering
# =====================

# Passenger Group
df['Group'] = df['PassengerId'].str.split('_').str[0].astype(int)
df['GroupSize'] = df.groupby('Group')['Group'].transform('count')

# Cabin split
df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)

# FIX: Convert CabinNum to numeric
df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')

# Spending
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df[spend_cols] = df[spend_cols].fillna(0)
df['TotalSpend'] = df[spend_cols].sum(axis=1)

# CryoSleep logic
df.loc[df['CryoSleep'] == True, spend_cols + ['TotalSpend']] = 0

# Name feature
df['Surname'] = df['Name'].str.split().str[-1]

# =====================
# Missing Values
# =====================
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'Surname']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['CabinNum'] = df['CabinNum'].fillna(-1)

# =====================
# Encode Categoricals
# =====================
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# =====================
# Features
# =====================
features = [
    'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
    'Deck', 'CabinNum', 'Side',
    'GroupSize', 'TotalSpend'
]

# =====================
# Split Back
# =====================
train_df = df[df['is_train'] == 1]
test_df  = df[df['is_train'] == 0]

X = train_df[features]
y = train_df['Transported'].astype(int)
X_test = test_df[features]

# =====================
# Train / Validation
# =====================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =====================
# Model
# =====================
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# =====================
# Validation
# =====================
val_preds = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

# =====================
# Train Full + Predict
# =====================
model.fit(X, y)
test_preds = model.predict(X_test).astype(bool)

# =====================
# Submission
# =====================
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': test_preds
})

submission.to_csv('submission.csv', index=False)
print("submission.csv saved")
submission.head()

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001933 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
Validation Accuracy: 0.7527314548591144
[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000367 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 624
[LightGBM] [Info] Number of data points in the train set: 8693, number of used f

,PassengerId,Transported
8693,0013_01,True
8694,0018_01,False
8695,0019_01,True
8696,0021_01,False
8697,0023_01,False
